In [33]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [48]:
df = pd.read_csv('/content/drive/MyDrive/softmax_data(Sheet1).csv')

# Convert 'cgpa' and 'iq' columns to numeric, coercing errors to NaN
df['cgpa'] = pd.to_numeric(df['cgpa'], errors='coerce')
df['iq'] = pd.to_numeric(df['iq'], errors='coerce')

# Drop rows with NaN values that resulted from coercion
df.dropna(inplace=True)

Cleaned 'cgpa' and 'iq' columns in df by converting to numeric and dropping rows with non-convertible values.


In [49]:
X = df.drop('placement', axis=1)
y = df['placement']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

Data successfully re-split into training and testing sets after cleaning.


In [50]:
class softmax:
  def __init__(self, n_features, n_classes):
    self.m = np.zeros((n_features, n_classes))  # Initialize weights as a matrix
    self.b = np.zeros(n_classes)              # Initialize biases as a vector
    self.n_classes = n_classes

  def _one_hot_encode(self, y, n_classes):
    # Create a NumPy array of zeros with dimensions (len(y), n_classes)
    one_hot_matrix = np.zeros((len(y), n_classes))
    # Iterate through the y array and set the appropriate element to 1
    for i, label in enumerate(y):
      one_hot_matrix[i, int(label)] = 1
    return one_hot_matrix

  def logit(self, X):
    # Perform matrix multiplication for weighted sum and add bias
    # Ensure X is a numpy array before multiplication
    z = X @ self.m + self.b
    return z

  def subtract_max(self, z):
    # Ensure stability by subtracting the maximum value from z
    z_stable = z - np.max(z, axis=1, keepdims=True)
    return z_stable

  def exponential(self, z_stable):
    exp = np.exp(z_stable)
    return exp

  def softmax_function(self, exp):
    sum_exp = np.sum(exp, axis=1, keepdims=True)
    softmax = exp / sum_exp
    return softmax

  def loss(self, y_true, y_pred):
    # Compute cross-entropy loss
    N = y_true.shape[0]
    # Add a small epsilon to y_pred to prevent log(0)
    loss = -np.sum(y_true * np.log(y_pred + 1e-10)) / N
    return loss

  def compute_gradients(self, X, y_true_one_hot, softmax_probs):
    # Calculate the error term
    error = softmax_probs - y_true_one_hot

    # Number of samples
    N = X.shape[0]

    # Compute gradient for weights (m)
    grad_m = (X.T @ error) / N

    # Compute gradient for biases (b)
    grad_b = np.sum(error, axis=0) / N

    return grad_m, grad_b

  def fit(self, X_train, y_train, learning_rate=0.01, epochs=100):
    # Convert X_train to numpy array to avoid pandas specific issues
    X_train_np = X_train.to_numpy()
    # Convert y_train to one-hot encoded format
    y_train_one_hot = self._one_hot_encode(y_train, self.n_classes)

    for epoch in range(epochs):
      # Perform forward pass
      z = self.logit(X_train_np) # Changed X_train to X_train_np here
      z_stable = self.subtract_max(z)
      exp = self.exponential(z_stable)
      softmax_probs = self.softmax_function(exp)

      # Calculate loss
      current_loss = self.loss(y_train_one_hot, softmax_probs)

      # Compute gradients
      grad_m, grad_b = self.compute_gradients(X_train_np, y_train_one_hot, softmax_probs)

      # Update weights and biases
      self.m -= learning_rate * grad_m
      self.b -= learning_rate * grad_b

      # Periodically print loss
      if epoch % 10 == 0 or epoch == epochs - 1:
        print(f"Epoch {epoch}/{epochs}, Loss: {current_loss:.4f}")

  def predict(self, X_test):
    # Convert X_test to numpy array for consistent operations
    X_test_np = X_test.to_numpy()
    z = self.logit(X_test_np)
    z_stable = self.subtract_max(z)
    exp = self.exponential(z_stable)
    soft = self.softmax_function(exp)
    # Return the index of the class with the highest probability
    return np.argmax(soft, axis=1)


In [51]:
n_features = X_train.shape[1]
n_classes = len(y_train.unique())
s = softmax(n_features, n_classes)

In [52]:
s.fit(X_train, y_train)

Epoch 0/100, Loss: 1.0986
Epoch 10/100, Loss: 1.0295
Epoch 20/100, Loss: 1.0208
Epoch 30/100, Loss: 1.0154
Epoch 40/100, Loss: 1.0115
Epoch 50/100, Loss: 1.0086
Epoch 60/100, Loss: 1.0065
Epoch 70/100, Loss: 1.0048
Epoch 80/100, Loss: 1.0034
Epoch 90/100, Loss: 1.0022
Epoch 99/100, Loss: 1.0013
Softmax model training initiated with default learning rate and epochs.


In [53]:
from sklearn.metrics import accuracy_score,classification_report


In [54]:
y_pred = s.predict(X_test)

In [55]:
accuracy = accuracy_score(y_test, y_pred)
print(accuracy)
report = classification_report(y_test, y_pred)
print(report)

0.42857142857142855
              precision    recall  f1-score   support

           0       0.60      0.75      0.67         4
           1       0.00      0.00      0.00         2
           2       0.00      0.00      0.00         1

    accuracy                           0.43         7
   macro avg       0.20      0.25      0.22         7
weighted avg       0.34      0.43      0.38         7



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
